In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}
try:
    driver = webdriver.Chrome()
except:
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=chrome_options)
    print("Running in headless mode.")

In [3]:
base_url = 'https://igod.gov.in/sectors'

In [4]:
def safe_request(url):
    try:
        # Use requests to check the status code first
        response = requests.get(url)
        driver.get(url)
        time.sleep(5)  # Wait for the page to load
        last_height = driver.execute_script("return document.body.scrollHeight")
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(15)  # Short wait for new content to load
            new_height = driver.execute_script("return document.body.scrollHeight")
            print(last_height , new_height) 
            if new_height == last_height:
                break
            last_height = new_height
        print(f"{url} => Page retrieved successfully!")
        return driver.page_source , response.status_code
    except Exception as e:
        print(f"{url} => An error occurred: {e}")
        return None , response.status_code

In [5]:
# Send a GET request to fetch the HTML content
res , status = safe_request(base_url)
if res:
    soup = BeautifulSoup(res, 'html.parser')
    # Extract links
    sector_links = {a.text.strip().replace(" ","_").replace("_&" , ""): a['href'] for a in soup.select('.sector-container .sector-box')}
sector_links_slice = dict(list(sector_links.items()))
data = []
count = 1
for name , link in sector_links_slice.items():
    print(f'count => {count} out of {len(sector_links_slice)}')
    data.append({
        'sector_name':name,
        'sector_link': link
    })
    main_page_source , status = safe_request(link)
    if main_page_source:
        soup = BeautifulSoup(main_page_source, 'html.parser')
        # Find all search result rows
        search_rows = soup.find_all('div', class_=['search-result-row', 'search-result-row '])
        print(len(search_rows))
        sub_count = 1
        for row in search_rows:
            print(f'sub_count => {sub_count} out of {len(search_rows)}')
            sub_count += 1
            link = row.find('a', class_='search-title')
            
            if link and link['href']:
                link_url = link['href']
                title_text = link.get_text(strip=True)
                # if 200 <= status < 300:
                #     response = 'Page retrieved successfully'
                # elif response.status_code == 404:
                #     response = "Error: 404 Not Found"
                # elif response.status_code == 503:
                #     response = "Error: 503 Service Unavailable"
                # elif response.status_code >= 400:
                #     response = "Client Error"
                # elif response.status_code >= 500:
                #     response = "Server Error"
                data.append({
                    'sector_name':'',
                    'sector_link':'',
                    'count_links':len(search_rows),
                    'title':title_text,
                    'link': link_url,
                })
    count += 1
df = pd.DataFrame(data)
df.to_csv('sectors_data.csv', index=False)

# Cleanup: Close the Selenium driver
driver.quit()

2296 3118
3118 3118
https://igod.gov.in/sectors => Page retrieved successfully!
count => 1 out of 30
3401 9498
9498 11708
11708 11708
https://igod.gov.in/sector/GRNsIHQBsvhI6u6Q3tju/organizations => Page retrieved successfully!
257
sub_count => 1 out of 257
sub_count => 2 out of 257
sub_count => 3 out of 257
sub_count => 4 out of 257
sub_count => 5 out of 257
sub_count => 6 out of 257
sub_count => 7 out of 257
sub_count => 8 out of 257
sub_count => 9 out of 257
sub_count => 10 out of 257
sub_count => 11 out of 257
sub_count => 12 out of 257
sub_count => 13 out of 257
sub_count => 14 out of 257
sub_count => 15 out of 257
sub_count => 16 out of 257
sub_count => 17 out of 257
sub_count => 18 out of 257
sub_count => 19 out of 257
sub_count => 20 out of 257
sub_count => 21 out of 257
sub_count => 22 out of 257
sub_count => 23 out of 257
sub_count => 24 out of 257
sub_count => 25 out of 257
sub_count => 26 out of 257
sub_count => 27 out of 257
sub_count => 28 out of 257
sub_count => 29 out o